In [2]:
import tensorflow as tf
import os
import numpy as np
import matplotlib.pyplot as plt
import pathlib

In [7]:
def augement_data(image):
  image = tf.image.resize_with_crop_or_pad(image, 180,180)
  image = tf.image.random_crop(image, size=[150,150,3])
  image = tf.image.random_brightness(image,max_delta=0.5)

  return image

In [42]:
directory = tf.keras.utils.get_file(
    'flower_photos',
    'https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz',
    untar=True
)

In [43]:
train_dir = pathlib.Path(directory+'/flower_photos')
train_dir

PosixPath('/root/.keras/datasets/flower_photos/flower_photos')

In [ ]:
! ls /root/.keras/datasets/flower_photos/flower_photos/roses/99383371_37a5ac12a3_n.jpg

In [46]:
CLASS_NAMES = np.array([item.name for item in train_dir.glob('*') if item.name != 'LICENSE.txt'])

In [47]:
CLASS_NAMES

array(['tulips', 'daisy', 'sunflowers', 'roses', 'dandelion'],
      dtype='<U10')

In [48]:
full_dataset = tf.data.Dataset.list_files(str(train_dir/'*/*'))

In [49]:
validation_split = 0.2

In [57]:
DATASET_SIZE = len(list(full_dataset))
print(f"full dataset size: {DATASET_SIZE}")
train_dataset = full_dataset.take(int(DATASET_SIZE*(1-validation_split)))
validation_dataset = full_dataset.skip(int((1-validation_split)*DATASET_SIZE))
print(f"train data set size: {len(list(train_dataset))}")
print(f"validation data set size: {len(list(validation_dataset))}")

full dataset size: 3670
train data set size: 2936
validation data set size: 734


In [58]:
def get_label(file_path):
  parts = tf.strings.split(file_path, os.path.sep)
  return parts[-2] == CLASS_NAMES

In [60]:
get_label('/root/.keras/datasets/flower_photos/flower_photos/roses/99383371_37a5ac12a3_n.jpg')

<tf.Tensor: shape=(5,), dtype=bool, numpy=array([False, False, False,  True, False])>

In [61]:
def load_img(image_path):
  img = tf.io.read_file(image_path)
  img = tf.image.decode_image(img, 3, expand_animations=False)
  img = tf.cast(img, tf.float32)
  return img

In [62]:
def normalize(image):
  return ((image/127.5) - 1)

In [63]:
def resize(image, height, width):
  return tf.image.resize(image, (height,width), method = tf.image.ResizeMethod.NEAREST_NEIGHBOR)

In [64]:
def load_image_with_label(image_path):
  label = get_label(image_path)
  img = load_img(image_path)
  return img, label

In [65]:
def load_image_train(image_file):
  image, label = load_image_with_label(image_file)
  image = augement_data(image)
  image = normalize(image)

  return image, label

In [66]:
def load_image_test(image_file):
  image, label = load_image_with_label(image_file)
  image = resize(image, 150, 150)
  image = normalize(image)

  return image, label

In [67]:
BATCH_SIZE = 32
SUFFLE_BUFFER_SIZE = 1000

In [68]:
train_dataset = train_dataset.map(load_image_train)
train_dataset = train_dataset.shuffle(SUFFLE_BUFFER_SIZE)
train_dataset = train_dataset.batch(BATCH_SIZE)

In [69]:
validation_dataset = validation_dataset.map(load_image_test)
validation_dataset = validation_dataset.batch(BATCH_SIZE)

In [70]:
base_model = tf.keras.applications.VGG16(weights='imagenet', include_top=False, input_shape=(150,150,3))

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


In [71]:
base_model.trainable = False

In [72]:
n_class = len(CLASS_NAMES)

In [74]:
flatten_layer = tf.keras.layers.GlobalAveragePooling2D()
dense_layer = tf.keras.layers.Dense(100, activation='relu')
dropout_layer = tf.keras.layers.Dropout(0.5)
prediction_layer = tf.keras.layers.Dense(n_class, activation='softmax')

In [75]:
model = tf.keras.Sequential([
    base_model,
    flatten_layer,
    dense_layer,
    dropout_layer,
    prediction_layer
])

In [77]:
model.compile(optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.01),loss = tf.keras.losses.categorical_crossentropy, metrics=['accuracy'])

In [84]:
history = model.fit(train_dataset, epochs=100, validation_data=validation_dataset)

Epoch 1/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 37s 277ms/step - accuracy: 0.5153 - loss: 1.5203 - val_accuracy: 0.6308 - val_loss: 0.9721
Epoch 2/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 14s 129ms/step - accuracy: 0.7137 - loss: 0.7947 - val_accuracy: 0.7057 - val_loss: 0.8128
Epoch 3/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 14s 129ms/step - accuracy: 0.7389 - loss: 0.7026 - val_accuracy: 0.7425 - val_loss: 0.6627
Epoch 4/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 14s 136ms/step - accuracy: 0.7676 - loss: 0.6497 - val_accuracy: 0.7466 - val_loss: 0.7330
Epoch 5/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 14s 126ms/step - accuracy: 0.7655 - loss: 0.6252 - val_accuracy: 0.7575 - val_loss: 0.6638
Epoch 6/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 14s 126ms/step - accuracy: 0.7548 - loss: 0.6562 - val_accuracy: 0.7343 - val_loss: 0.6980
Epoch 7/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 14s 128ms/step - accuracy: 0.7796 - loss: 0.6365 - val_accuracy: 0.7398 - val_loss: 0.6635
Epoch 8/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 14s 127ms/step - accuracy: 0.7890 - loss: 0.6063 - 